In [1]:
import json, os, asyncio, pdb
import pandas as pd
from pathlib import Path
import pyprojroot, sys
from pyprojroot.criterion import has_file
root = pyprojroot.find_root(has_file("pyproject.toml"))

sys.path.insert(0, str(root))
os.chdir(root)  #
from dotenv import load_dotenv

In [2]:
load_dotenv()
from kdutils.macro import base_path


In [3]:
trade_date = "2026-10-16"

In [4]:
filename = os.path.join(base_path,"live", "temp", trade_date, "predict",
                        "{0}_000852.json".format(trade_date))

In [5]:
with open(filename, "r", encoding="utf-8") as f:
    snapshot = json.load(f)

In [6]:
snapshot['trader_prediction']

{'dominant_regime': '外部科技股风暴冲击引发高波急跌后的政策托底企稳期：估值分位、仓位箱体与利率均处历史低位，市场宽度单日快速修复、期指升水推向极值、政策宽松口头定调落地，属超跌反弹博弈窗口',
 'predict_direction': 'UP',
 'confidence': 0.58,
 'used_features': ['main_flow_ratio',
  'main_flow_acceleration',
  'basis_div_strength',
  'basis_diff_3d',
  'panic_reversal_score',
  'exhaustion_intensity',
  'TF-20261016-002',
  'TF-20261016-004',
  'TF-20261015-005'],
 'predict_logic': [{'feature': 'main_flow_ratio',
   'feature_layer': 'PREDICT',
   'observation': '10-15主力净流出触及-0.4110的窗口内最深值后，10-16单日急速反转为+0.3787的强净流入，单日摆幅接近0.79，资金面呈V型反转',
   'inference': "恐慌性抛售后主动买盘重新大举入场，短线反弹获得资金燃料；该读数恰好满足历史案例中'主力资金转正'这一重新做多的前置确认条件"},
  {'feature': 'main_flow_acceleration',
   'feature_layer': 'PREDICT',
   'observation': '10-16流量加速度仍为-0.2966负值，且momentum_vol_adj_20(-0.1875)、gap_continuation(-0.2802)、price_acc_5d(-0.1604)等趋势类信号均未转正',
   'inference': '资金回流尚属单日水平反转、动量趋势未获二次确认，上行斜率受限，故置信度仅给予中等水平而非高置信'},
  {'feature': 'basis_div_strength',
   'feature_layer': 'PREDICT',
   'observation': '基差背离强

In [10]:
from typing import Dict, Any, List

FEATURE_MAP = {
    "main_flow_acceleration": "主力资金加速度（二阶动量）",
    "main_flow_ratio": "主力资金净流入比率",
    "price_acc_5d": "5日价格动量与加速度",
    "breadth_div_intensity_pos": "市场宽度与正向背离",
    "oi_change_3d": "衍生品持仓量变动（3D OI）",
    "panic_reversal_score": "恐慌反转得分",
    "basis_div_strength": "期货基差背离强度",
    "global_liquidity": "全球流动性宏观环境",
    "domestic_policy": "国内政策对冲与托底",
    "external_shock":"外部冲击",
}


class TradingMemoGenerator:
    """策略研判与交易决策备忘录生成器（执行与风控先行版）"""

    def __init__(self, data: Dict[str, Any]):
        self.data = data
        self.direction = data.get("predict_direction", "NEUTRAL")
        self.confidence = data.get("confidence", 0.0)

    def _render_header(self) -> str:
        dir_badge = {
            "DOWN": "偏空（DOWN）🔻",
            "UP": "偏多（UP）🔺",
            "NEUTRAL": "中性观望（NEUTRAL）⚪",
        }.get(self.direction, self.direction)

        conf_tag = (
            "高置信度"
            if self.confidence >= 0.75
            else "中等置信度"
            if self.confidence >= 0.55
            else "低置信度"
        )

        return f"""# 策略研判与交易决策备忘录

* **交易基准日**：`{self.data.get('trade_time', 'N/A')}`
* **研判模块**：`{self.data.get('name', 'N/A')}`
* **核心方向**：**{dir_badge}**
* **置信度**：`{self.confidence:.2f}`（{conf_tag}）
"""

    def _render_execution_summary(self) -> str:
        """一、交易指令摘要（提前至首位）"""
        if self.direction == "DOWN":
            stance = "轻仓试空或收缩多头仓位；由于政策对冲存在，**严禁激进追空**。"
        elif self.direction == "UP":
            stance = "逢低偏多布局，警惕上方压力。"
        else:
            stance = "多空方向不明，维持区间中性交易。"

        return f"""### 一、 交易指令摘要（Execution Summary）

* **战术头寸建议**：{stance}
* **核心风控盯盘**：重点关注主力资金是否二阶回正、基差背离强弱及政策落地长阳催化。
"""

    def _render_invalid_conditions(self) -> str:
        """二、风控与逻辑失效边界（提前至状态前）"""
        conds = self.data.get("invalid_conditions", [])
        if not conds:
            return ""

        lines = [
            "### 二、 风控与逻辑失效边界（Invalid Conditions）\n",
            "交易执行必须严格遵守以下**失效平仓/止损与反手预案**：\n",
        ]
        for i, cond in enumerate(conds, start=1):
            lines.append(f"{i}. {cond}")
        lines.append("")
        return "\n".join(lines)

    def _render_regime(self) -> str:
        """三、市场主导状态"""
        regime = self.data.get("dominant_regime", "暂无定性描述")
        return f"""### 三、 市场主导状态（Dominant Regime）

> **{regime}**
"""

    def _render_predict_logic(self) -> str:
        """四、多维量化信号与逻辑推演"""
        logics: List[Dict[str, Any]] = self.data.get("predict_logic", [])
        if not logics:
            return ""

        quant_signals = [l for l in logics if l.get("feature_layer") == "PREDICT"]
        macro_signals = [l for l in logics if l.get("feature_layer") == "TEXTUAL"]

        lines = ["### 四、 多维量化信号与逻辑推演（Predict Logic）\n"]

        if quant_signals:
            lines.append("#### 1. 量化微观结构与衍生品特征")
            for item in quant_signals:
                feat = item.get("feature", "")
                title = FEATURE_MAP.get(feat, feat)
                obs = item.get("observation", "")
                inf = item.get("inference", "")
                lines.append(f"- **{title}**：{obs}\n  - *推演逻辑*：{inf}")
            lines.append("")

        if macro_signals:
            lines.append("#### 2. 宏观流动性与政策文本催化（Textual Factors）")
            for item in macro_signals:
                feat = item.get("feature", "")
                title = FEATURE_MAP.get(feat, feat)
                obs = item.get("observation", "")
                inf = item.get("inference", "")
                lines.append(f"- **{title}**：{obs}\n  - *推演逻辑*：{inf}")
            lines.append("")

        return "\n".join(lines)

    def generate(self) -> str:
        """组合生成完整 Markdown（新顺序：指令摘要 -> 风控边界 -> 主导状态 -> 信号推演）"""
        sections = [
            self._render_header(),
            self._render_execution_summary(),   # 1. 交易指令摘要
            self._render_invalid_conditions(),  # 2. 风控与失效边界
            self._render_regime(),              # 3. 市场主导状态
            self._render_predict_logic(),       # 4. 信号逻辑推演
        ]
        return "\n".join(s for s in sections if s)

In [11]:
generator = TradingMemoGenerator(snapshot['trader_prediction'])
markdown_output = generator.generate()
from rich.console import Console
from rich.markdown import Markdown

In [12]:
from rich.console import Console
from rich.markdown import Markdown
from rich.theme import Theme

# 1. 定义自定义主题，将标题和引用块强制改为白色
custom_theme = Theme({
    "markdown.h1": "bold white",           # 一级标题
    "markdown.h2": "bold white",           # 二级标题
    "markdown.h3": "bold white",           # 三级标题（如：一、市场主导状态）
    "markdown.h4": "bold white",           # 四级标题
    "markdown.block_quote": "bright_white",# 引用块竖线与内容（即图中的 ▌ 区域）
    "markdown.paragraph": "white",         # 正文段落
    "markdown.text": "white",              # 普通文本
    "markdown.item": "white",              # 列表项
})

# 2. 传入 theme 初始化 Console
console = Console(theme=custom_theme)

# 3. 正常输出即可变为纯白 / 亮白色
console.print(Markdown(markdown_output))

策略研判与交易决策备忘录                                              

 • 交易基准日：2026-10-16                                                                                          
 • 研判模块：reflection1                                                                                           
 • 核心方向：偏多（UP）🔺                                                                                          
 • 置信度：0.58（中等置信度）                                                                                      

一、 交易指令摘要（Execution Summary）                                                                             

 • 战术头寸建议：逢低偏多布局，警惕上方压力。                                                                      
 • 核心风控盯盘：重点关注主力资金是否二阶回正、基差背离强弱及政策落地长阳催化。                                    

二、 风控与逻辑失效边界（Invalid Conditions）                                                                      

交易执行必须严格遵守以下失效平仓/止损与反手预案：                                                                  

 1 T+1盘中主力资金再度转净流出（main_flow_ratio回吐至0下方）且市场宽度分位重新跌破0.3，确认单日回流为诱多陷阱，反弹
   逻辑彻底失效，立即离场                                                                                          
 2 期指升水结构快速回吐：basis_div_strength跌破0.2或basis_percentile自1.0极值显著回落，衍生品先行多头撤离，市场结构
   确认链断裂，多头逻辑失效                                                                                        
 3 全球科技股风暴二次升级（再度出现熔断级下跌或AI/半导体抛售向A股蔓延）或美债收益率再度飙升引发加息预期重燃，系统性
   风险压制下风控优先、无条件降仓                                                                                  
 4 10-19公布的前三季度GDP等宏观数据显著低于预期，政策托底叙事被证伪，多头逻辑失去事件面支撑                        

三、 市场主导状态（Dominant Regime）                                                                               

▌ 外部科技股风暴冲击引发高波急跌后的政策托底企稳期：估值分位、仓位箱体与利率均处历史低位，市场宽度单日快速修复、 
▌ 期指升水推向极值、政策宽松口头定调落地，属超跌反弹博弈窗口                                                     

四、 多维量化信号与逻辑推演（Predict Logic）                                                                       

1. 量化微观结构与衍生品特征                                                                                        

 • 主力资金净流入比率：10-15主力净流出触及-0.4110的窗口内最深值后，10-16单日急速反转为+0.3787的强净流入，单日摆幅接
   近0.79，资金面呈V型反转                                                                                         
    • 推演逻辑：恐慌性抛售后主动买盘重新大举入场，短线反弹获得资金燃料；该读数恰好满足历史案例中'主力资金转正'这一 
      重新做多的前置确认条件                                                                                       
 • 主力资金加速度（二阶动量）：10-16流量加速度仍为-0.2966负值，且momentum_vol_adj_20(-0.1875)、gap_continuation(-0.
   2802)、price_acc_5d(-0.1604)等趋势类信号均未转正                                                                
    • 推演逻辑：资金回流尚属单日水平反转、动量趋势未获二次确认，上行斜率受限，故置信度仅给予中等水平而非高置信     
 • 期货基差背离强度：基差背离强度连续三日维持0.4236-0.4286高位（10-16为0.4283），基差3日差分+0.3322为正，basis_perc
   entile_252升至1.0极值                                                                                           
    • 推演逻辑：期货端持续强于现货端，衍生品杠杆多头在急跌中提前定价反弹，市场结构出现'聪明钱'先行确认，与2025-04-0
      7政策底反弹前的基差形态同构                                                                                  
 • 恐慌反转得分：恐慌反转得分由10-15的-0.1565转正至+0.1775，抛售衰竭强度-0.1325持续处于负值卖压衰竭区，price_ret_5d
   由-0.4281收窄至-0.3454                                                                                          
    • 推演逻辑：单边下跌动能进入衰竭减速段，超卖反抽的微观技术条件已经具备                                         

2. 宏观流动性与政策文本催化（Textual Factors）                                                                     

 • 国内政策对冲与托底：TF-20261016-002：央行行长明确'继续适度宽松'、财政部长表态'更加积极财政'，confirmed级别政策定
   调于急跌当日落地                                                                                                
    • 推演逻辑：构成2025-04-07式'政策底'的弱化版先验：虽无国家队真金白银增持，但口头宽松明示叠加基差升水与主力回流 
      的EV2类市场结构双重确认，足以支撑短线偏多而非维持观望                                                        
 • 全球流动性宏观环境：TF-20261016-004：沃什讲话后市场对美联储加息概率预期下降；TF-20261016-006：IEA释放逾2000万桶 
   储备缓解能源通胀压力                                                                                            
    • 推演逻辑：外部流动性收紧与通胀压力的边际方向转向缓和，外盘利空最紧时点大概率已过                             
 • 外部冲击：TF-20261015-00